# Flow Matching → MNIST 生成（自力実装の最終目標）

**主教材:** Lipman et al., [Flow Matching Guide and Code](https://arxiv.org/abs/2412.06264)（§2–4）

これは **学習済みモデルを使わず、MNIST からFlow Matchingを学習して新しい数字を生成する** 最小構成です。Guideの2次元Code 1を、28×28画像に拡張するためのゴールコードとして扱ってください。画像用の小型U-Netは追加実装です（GuideのCode 1そのものではありません）。

実装を読む順序：`データ → 経路 → 速度場ネットワーク → 損失・学習 → ODE・生成 → 検証`。

**前提と定義**

- `X₁`：MNISTから得た画像。形 `(B, 1, 28, 28)`、値域 `[-1,1]`
- `X₀ ~ N(0,I)`：同じ形の独立な正規乱数
- `t ~ Uniform[0,1]`：サンプルごとに独立な時刻
- `Xₜ = (1-t)X₀ + tX₁`：直線補間による確率経路
- `U = X₁ - X₀`：各組に対する教師速度
- `vθ(Xₜ,t)`：速度を予測するニューラルネット
- `L(θ) = E[‖vθ(Xₜ,t) - U‖²]`：教師速度との二乗誤差（コードでは全要素で平均）
- 生成時：`x(0) ~ N(0,I)` から `dx/dt = vθ(x,t)` をオイラー法で積分

> ノートブックの学習部分は Colab でGPUが使えるときの実行を想定。無料版のGPU割り当て・所要時間・生成品質は保証されません。

## 1. 準備・MNISTデータ

In [ ]:
import math
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# torchvision.transforms.ToTensor() で [0,1]、Normalize で [-1,1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_data = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
loader = DataLoader(train_data, batch_size=128, shuffle=True,
                    num_workers=2, pin_memory=(device.type == "cuda"))

x1, _ = next(iter(loader))
print("MNIST batch:", x1.shape, "min/max:", x1.min().item(), x1.max().item())

plt.figure(figsize=(6, 3))
plt.imshow(make_grid(((x1[:16] + 1) / 2), nrow=8).permute(1, 2, 0))
plt.axis("off")
plt.title("Training images (MNIST)")
plt.show()

## 2. 確率経路・教師速度（Guideの中心式）

In [ ]:
def sample_path(x1: torch.Tensor):
    # x1: (B,1,28,28)  →  tは各画像で1個だけ引く
    x0 = torch.randn_like(x1)
    t = torch.rand(x1.shape[0], device=x1.device)
    t4 = t[:, None, None, None]  # (B,1,1,1) にして画像へbroadcast
    xt = (1 - t4) * x0 + t4 * x1
    target_velocity = x1 - x0
    return t, xt, target_velocity

# 数学と形が一致することを確認
_t, _xt, _target = sample_path(x1[:4].to(device))
assert _xt.shape == _target.shape == (4, 1, 28, 28)
assert _t.shape == (4,)
print("path tensor shapes OK")

## 3. 時刻条件付きの小型U-Netを自作

GuideのCode 1では2次元の入力をMLPで処理しました。MNISTでは画像中の局所的な構造を使うため、小さな **畳み込みU-Net** に置き換えます。**確率経路と損失関数はそのまま**です。

ネットワークは `vθ(xt,t)` を受け、入力画像と同じ `(B,1,28,28)` の速度を返します。

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, padding=1)
        self.norm1 = nn.GroupNorm(8, out_channels)
        self.time_proj = nn.Linear(time_dim, out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.skip = (nn.Identity() if in_channels == out_channels
                     else nn.Conv2d(in_channels, out_channels, 1))

    def forward(self, x, temb):
        h = F.silu(self.norm1(self.conv1(x)))
        h = h + self.time_proj(temb)[:, :, None, None]
        h = self.norm2(self.conv2(F.silu(h)))
        return (self.skip(x) + h) / math.sqrt(2.0)


class TinyUNet(nn.Module):
    def __init__(self, base=32, time_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            nn.Linear(32, time_dim), nn.SiLU(),
            nn.Linear(time_dim, time_dim),
        )
        self.enc1 = ResBlock(1, base, time_dim)          # 28 x 28
        self.enc2 = ResBlock(base, base * 2, time_dim)   # 14 x 14
        self.mid = ResBlock(base * 2, base * 4, time_dim)  # 7 x 7
        self.dec2 = ResBlock(base * 6, base * 2, time_dim) # 14 x 14
        self.dec1 = ResBlock(base * 3, base, time_dim)     # 28 x 28
        self.output = nn.Conv2d(base, 1, 3, padding=1)

    @staticmethod
    def time_features(t):
        # t : (B,)。連続時刻を32次元の特徴に変換する
        frequencies = torch.exp(torch.linspace(0, math.log(100), 16, device=t.device))
        phases = (2 * math.pi) * t[:, None] * frequencies[None, :]
        return torch.cat([torch.sin(phases), torch.cos(phases)], dim=1)

    def forward(self, xt, t):
        temb = self.time_mlp(self.time_features(t))
        s1 = self.enc1(xt, temb)
        s2 = self.enc2(F.avg_pool2d(s1, 2), temb)
        h = self.mid(F.avg_pool2d(s2, 2), temb)
        h = F.interpolate(h, size=s2.shape[-2:], mode="nearest")
        h = self.dec2(torch.cat([h, s2], dim=1), temb)
        h = F.interpolate(h, size=s1.shape[-2:], mode="nearest")
        h = self.dec1(torch.cat([h, s1], dim=1), temb)
        return self.output(F.silu(h))

model = TinyUNet(base=32).to(device)
parameters = sum(p.numel() for p in model.parameters())
print("Trainable parameters:", f"{parameters:,}")

# 事前テスト：予測するのは画像そのものではなく、画像と同形の速度
probe = model(_xt, _t)
assert probe.shape == _xt.shape, (probe.shape, _xt.shape)
print("model output shape OK", probe.shape)

## 4. Flow Matchingの学習（Guideと同じMSE）

In [ ]:
# 最初は 8 epoch を目安に。数字が不鮮明なら12〜20 epochなどに増やして検証。
# GPU時間が不足するときは epoch を減らして実行確認から進める。
EPOCHS = 8
LEARNING_RATE = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
loss_history = []

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    n_images = 0
    for x1, _ in loader:
        x1 = x1.to(device, non_blocking=True)
        t, xt, target = sample_path(x1)
        predicted = model(xt, t)
        loss = F.mse_loss(predicted, target)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * x1.shape[0]
        n_images += x1.shape[0]
    avg = total_loss / n_images
    loss_history.append(avg)
    print(f"epoch {epoch+1:02d}/{EPOCHS}: loss={avg:.5f}")

    # 無料Colabの切断に備え、各epochで /content にチェックポイント保存
    # Drive等にコピーしない場合はランタイム終了後に消えます。
    torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                "epoch": epoch + 1, "loss_history": loss_history},
               "flow_matching_mnist_checkpoint.pt")

plt.figure(figsize=(5, 3))
plt.plot(range(1, len(loss_history) + 1), loss_history, marker="o")
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.title("Flow Matching training")
plt.grid(alpha=0.25)
plt.show()

## 5. ODEをオイラー法で積分して画像を生成

学習後に新しいノイズ `x ~ N(0,I)` を引き、時刻0から1まで

\[x_{k+1}=x_k+\frac1N v_\theta\left(x_k,\frac{k}{N}\right)\]

で計算します。`n_steps` は生成過程で評価する速度場の回数です。画像表示用だけに値を `[-1,1]` へクリップします。

In [ ]:
@torch.no_grad()
def generate(model, n_images=64, n_steps=50, x0=None):
    model.eval()
    x = (torch.randn(n_images, 1, 28, 28, device=device)
         if x0 is None else x0.clone().to(device))
    dt = 1.0 / n_steps
    for k in range(n_steps):
        t = torch.full((x.shape[0],), k / n_steps, device=device)
        x = x + dt * model(x, t)
    return x

samples = generate(model, n_images=64, n_steps=50)
images = ((samples.clamp(-1, 1) + 1) / 2).cpu()

plt.figure(figsize=(8, 8))
plt.imshow(make_grid(images, nrow=8, padding=1).permute(1, 2, 0))
plt.title("MNIST generated by Flow Matching (unconditional)")
plt.axis("off")
plt.show()

## 6. 生成ステップ数の比較実験（同じ初期ノイズで比較）

In [ ]:
initial_noise = torch.randn(16, 1, 28, 28, device=device)
steps_to_compare = [5, 10, 20, 50]

fig, axes = plt.subplots(len(steps_to_compare), 1, figsize=(9, 9))
for ax, n_steps in zip(axes, steps_to_compare):
    result = generate(model, x0=initial_noise, n_steps=n_steps)
    img = ((result.clamp(-1, 1) + 1) / 2).cpu()
    ax.imshow(make_grid(img, nrow=8).permute(1, 2, 0))
    ax.set_title(f"Euler steps = {n_steps}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## 7. 到達基準と注意

**成功と判断する条件**
1. `vθ(xt,t)` の出力が `(B,1,28,28)` である。
2. MNISTの画像と正規ノイズから `xt` と教師速度を算出できる。
3. 学習のMSEがある程度下がる（MSEだけでは品質保証にならない）。
4. 事前学習済みモデルを使わず、ノイズから数字らしい画像が生成される。
5. 生成ステップ数を変更し、同じノイズからの結果を比較・考察できる。

**注意**：条件なし生成なので数字を指定することはできません。`-1〜1` の画像正規化やU-Netの実装は、原論文の一般理論とは別の実装上の選択です。全epochの学習品質は実行環境や学習時間に依存します。

次の発展は `オイラー法 vs 中点法`、`ノイズ分布・経路の変更`、`MNISTクラス条件付き生成` ですが、最初はここまで完成させることを目標にしてください。